### Params

In [4]:
seq = "053"

In [5]:
!ls -l outputs/full_split/neurad/

total 32
drwxr-xr-x 3 jovyan jovyan 4096 Oct  8 21:49 2025-10-08_214119
drwxr-xr-x 3 jovyan jovyan 4096 Oct  8 21:49 2025-10-08_214327
drwxr-xr-x 3 jovyan jovyan 4096 Oct  8 22:33 2025-10-08_222800
drwxr-xr-x 3 jovyan jovyan 4096 Oct  8 22:50 2025-10-08_224305
drwxr-xr-x 3 jovyan jovyan 4096 Oct  8 23:21 2025-10-08_231503
drwxr-xr-x 3 jovyan jovyan 4096 Oct  8 23:50 2025-10-08_234202
drwxr-xr-x 3 jovyan jovyan 4096 Oct  9 00:15 2025-10-09_000854
drwxr-xr-x 3 jovyan jovyan 4096 Oct  9 00:51 2025-10-09_004358


In [6]:
seq2folder = {
    "001": "2025-10-08_214119",
    "053": "2025-10-08_214327",
    # "063": "2025-10-08_222800",
    "011": "2025-10-08_224305",
    "106": "2025-10-08_231503",
    "016": "2025-10-08_234202",
    "123": "2025-10-09_000854",
    "028": "2025-10-09_004358"
}

In [7]:
folder = seq2folder[seq]
cfg = f"outputs/full_split/neurad/{folder}/config.yml"
shift = [-3.0, 0.0, 0.0]

str_shift = f"{shift[0]} {shift[1]} {shift[2]}"

### Rendering

In [107]:
!python nerfstudio/scripts/render_shifted.py --load-config {cfg} --output_path neurad_shifted/{seq} --shift {str_shift} --render_point_clouds False

2025-10-05 20:23:53.131949: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-05 20:23:53.276678: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-05 20:23:54.642493: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib:/home/jovyan/users/npata

### Video check

In [20]:
selected_cameras = ["left_camera", "front_camera"]
base_folder = f"shifted/{seq}/test"
fps = 10

In [21]:
import imageio
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
import glob
import os
import numpy as np

def create_video_with_imageio(base_folder, camera_name, output_path, fps=10, reverse=False):
    # Get image paths
    if camera_name == "lidar":
        rgb_folder = Path(base_folder)
        gt_rgb_folder = Path(base_folder)
        rgb_camera_path = os.path.join(rgb_folder, "lidar_vis" )
        gt_rgb_camera_path = os.path.join(gt_rgb_folder, "lidar_vis_gt")
        rgb_images = sorted(glob.glob(os.path.join(rgb_camera_path, "*.png")))
        gt_rgb_images = sorted(glob.glob(os.path.join(gt_rgb_camera_path, "*.png")))
    else:
        rgb_folder = Path(base_folder) / "rgb" 
        gt_rgb_folder = Path(base_folder) / "gt-rgb"
        rgb_camera_path = os.path.join(rgb_folder, camera_name)
        gt_rgb_camera_path = os.path.join(gt_rgb_folder, camera_name)
        rgb_images = sorted(glob.glob(os.path.join(rgb_camera_path, "*.jpg")))
        gt_rgb_images = sorted(glob.glob(os.path.join(gt_rgb_camera_path, "*.jpg")))
        
    frames = []
    for i, (rgb_path, gt_rgb_path) in enumerate(zip(rgb_images, gt_rgb_images)):
        # Read images using PIL
        rgb_img = Image.open(rgb_path)
        gt_rgb_img = Image.open(gt_rgb_path)
        
        # Get dimensions
        w, h = rgb_img.size

        w = int(w * 0.4)
        h = int(h * 0.4)
        
        # Resize GT image to match RGB if needed
        gt_rgb_img = gt_rgb_img.resize((w, h))
        rgb_img = rgb_img.resize((w, h))
        
        # Create side-by-side image
        combined_img = Image.new('RGB', (w * 2, h))
        combined_img.paste(rgb_img, (0, 0))
        combined_img.paste(gt_rgb_img, (w, 0))
        
        # Add text labels using PIL
        draw = ImageDraw.Draw(combined_img)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20)
        except:
            font = ImageFont.load_default()
        
        # Add labels
        if camera_name != "lidar":
            if reverse:
                draw.text((10, 10), f'Reverse shift', fill=(255, 255, 255), font=font)
                draw.text((w + 10, 10), f'Ground Truth with shift {shift}', fill=(255, 255, 255), font=font)
            else:
                draw.text((10, 10), f'Shift {shift}', fill=(255, 255, 255), font=font)
                draw.text((w + 10, 10), 'Ground Truth', fill=(255, 255, 255), font=font)

        frame_array = np.array(combined_img)
        frames.append(frame_array)
    imageio.mimsave(output_path, frames, fps=fps)

In [110]:
(Path(base_folder) / "videos").mkdir(parents=True, exist_ok=True)

for selected_camera in selected_cameras:
    output_video_path = Path(base_folder) / "videos" / f"{selected_camera}.mp4"
    create_video_with_imageio(base_folder, selected_camera, output_video_path, fps)

### Final dataset

In [91]:
!CUDA_VISIBLE_DEVICES=0 python nerfstudio/scripts/render_shifted_neurad.py --load-config {cfg} --output_path neurad_shifted/{seq} --shift {str_shift} --render_point_clouds True

2025-10-10 14:44:02.404672: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-10 14:44:02.550920: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-10 14:44:03.915240: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib:/usr/local/cuda-12.4/lib

In [92]:
!cp -r /workspace/datasets/self-driving/pandaset/{seq} data/pandaset/

In [93]:
!cp -r neurad_shifted/{seq}/test/lidar/* data/pandaset/{seq}/lidar

In [94]:
!cp -r neurad_shifted/{seq}/test/rgb/front_camera/* data/pandaset/{seq}/camera/front_camera

In [95]:
!cp -r neurad_shifted/{seq}/test/rgb/left_camera/* data/pandaset/{seq}/camera/left_camera

In [96]:
!cp -r neurad_shifted/{seq}/test/rgb/right_camera/* data/pandaset/{seq}/camera/right_camera

In [97]:
!cp -r neurad_shifted/{seq}/test/rgb/back_camera/* data/pandaset/{seq}/camera/back_camera

In [98]:
!cp -r neurad_shifted/{seq}/test/rgb/front_left_camera/* data/pandaset/{seq}/camera/front_left_camera

In [99]:
!cp -r neurad_shifted/{seq}/test/rgb/front_right_camera/* data/pandaset/{seq}/camera/front_right_camera

### Optimize

In [22]:
#CUDA_VISIBLE_DEVICES=0 python nerfstudio/scripts/train.py splatad --experiment_name="shifts" pandaset-data --sequence {seq}

### Test

In [8]:
!ls -l outputs/cycle/neurad/

total 28
drwxr-xr-x 3 jovyan jovyan 4096 Oct 10 15:15 2025-10-10_150816
drwxr-xr-x 3 jovyan jovyan 4096 Oct 10 15:16 2025-10-10_150819
drwxr-xr-x 3 jovyan jovyan 4096 Oct 10 16:05 2025-10-10_155746
drwxr-xr-x 3 jovyan jovyan 4096 Oct 10 16:19 2025-10-10_161105
drwxr-xr-x 3 jovyan jovyan 4096 Oct 10 17:04 2025-10-10_165631
drwxr-xr-x 3 jovyan jovyan 4096 Oct 10 17:18 2025-10-10_171013
drwxr-xr-x 3 jovyan jovyan 4096 Oct 10 18:19 2025-10-10_181203


In [ ]:
# 106, 011, 123, 016, 028

In [43]:
seq = "028"

In [44]:
seq2file = {
    "001": "2025-10-10_150816",
    "001": "2025-10-10_150819",
    "106": "2025-10-10_155746",
    "011": "2025-10-10_161105",
    "123": "2025-10-10_165631",
    "016": "2025-10-10_171013",
    "028": "2025-10-10_181203"
}

In [45]:
str_shift = f"{-shift[0]} {-shift[1]} {-shift[2]}"

folder = seq2file[seq]
cfg = f"outputs/cycle/neurad/{folder}/config.yml"
seq

'028'

In [46]:
!python nerfstudio/scripts/render_shifted_neurad.py --load-config {cfg} --output_path neurad_reverse_shifted/{seq} --shift {str_shift} --render_point_clouds False --data data/pandaset

2025-10-11 02:27:35.200699: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-11 02:27:35.345859: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-11 02:27:36.871244: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib:/usr/local/cuda-12.4/lib

In [47]:
base_folder = f"neurad_reverse_shifted/{seq}/test"
(Path(base_folder) / "videos").mkdir(parents=True, exist_ok=True)

print(base_folder)
for selected_camera in selected_cameras:
    output_video_path = Path(base_folder) / "videos" / f"{selected_camera}.mp4"
    create_video_with_imageio(base_folder, selected_camera, output_video_path, fps, reverse=True)

neurad_reverse_shifted/028/test


In [23]:
base_folder = f"neurad_reverse_shifted/{seq}/test"
(Path(base_folder) / "videos").mkdir(parents=True, exist_ok=True)

output_video_path = Path(base_folder) / "videos" / f"lidar_new.mp4"
create_video_with_imageio(base_folder, "lidar", output_video_path, fps, reverse=True)

### Compute metrics

In [48]:
!CUDA_VISIBLE_DEVICES=0 python nerfstudio/scripts/eval.py --load-config {cfg} --data-root-path /workspace/datasets/self-driving/pandaset --output-path metrics/cycle_pandaset/neurad/{seq}.json

2025-10-11 02:31:19.308377: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-11 02:31:19.464579: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-11 02:31:20.905313: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda-12.4/lib64:
2025-10-11 